# Module 1: Build the Graph

An LLM converts five hotel FAQ documents into a typed knowledge graph. `SimpleKGPipeline`,
from the `neo4j-graphrag` package, runs that conversion. These five hotels were reserved for
you to add. They become a permanent part of the graph. The remaining modules query them.

An embedding places text near other text that means something similar. It does not record
that a hotel offers a spa, that the spa costs extra, or that the hotel sits in Cairo.
Extraction writes those facts down as nodes and relationships, so a query can match them
exactly instead of inferring them from nearby text. This module writes both, and every
module after it reads both.

## What this module does

| Step | What happens |
|------|--------------|
| Extraction | `SimpleKGPipeline` reads five documents and writes `Hotel`, `Room`, `Amenity`, `Policy`, and `Service` nodes |
| Schema | A fixed schema keeps labels consistent across documents |
| Schema comparison | An optional extraction without a schema shows how labels can vary |
| Indexes | The module creates vector and full-text indexes for the new chunk embeddings |
| Verification | The module checks the graph requirements used by later modules |

## Why only five documents?

The source archive holds the workshop hotel FAQ corpus. The dump you restored during Setup
contains the preloaded documents, extracted under the same schema this module pins. Building
the full corpus takes hours, so five documents were held back for you. They take about four minutes.

The five hotels you extract become ordinary nodes in the same graph as the preloaded hotels.
Later modules run retrieval against the combined graph.

## The two layers of the graph

Extraction writes two connected layers.

**The lexical layer holds the text.** Each source file becomes one `Document` node. The text
is split into chunks, and each chunk becomes a `Chunk` node carrying that text and a
1024-dimension embedding of it. Vector search and keyword search read this layer.

**The domain layer holds the facts stated in that text.** A `Hotel` node carries the name,
address, and guest rating as properties. `Room`, `Amenity`, `Policy`, and `Service` nodes hang
off it through typed relationships. Cypher queries read this layer.

`FROM_CHUNK` and `FROM_DOCUMENT` join the two layers, which is what makes graph retrieval
work. A search lands on a chunk, and the traversal from that chunk reaches typed facts and the
document they came from.

```text
hotel-tokyo-002.txt
    |  one source file, about 7 KB of text
    v
(:Document {source_filename: "hotel-tokyo-002.txt"})
    ^
    |  FROM_DOCUMENT                      the lexical layer: the text
(:Chunk {text, embedding: 1024 floats})
    ^
    |  FROM_CHUNK                         the domain layer: the facts
(:Hotel {name, address, guest_rating, total_rooms, email, phone})
    |
    +-[:HAS_ROOM]---------> (:Room {type, bed_configuration, max_occupancy, min_rate})
    +-[:OFFERS_AMENITY]---> (:Amenity {name, description, fee})
    +-[:HAS_POLICY]-------> (:Policy {name, description})
    +-[:PROVIDES_SERVICE]-> (:Service {name, description, cost, hours})
```

Every relationship starts at `Hotel`, so each document produces a small star of facts one hop
deep.

## Setup

This cell adds two directories to the Python import path. The `notebooks/` directory contains
the shared `workshop` package. The `02-connected-context` directory holds
`graph_builder.py`, the extraction code this notebook calls. Module 1 and Module 2 share that
code, so it lives in one directory rather than two.

In [ ]:
import os
import sys

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

# The shared `workshop` package at the notebooks/ root.
_notebooks_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _notebooks_root not in sys.path:
    sys.path.insert(0, _notebooks_root)

# graph_builder.py and graph_config.py, which build and verify the graph.
_build_machinery = os.path.join(_notebooks_root, "02-connected-context")
if _build_machinery not in sys.path:
    sys.path.insert(0, _build_machinery)

In [ ]:
# Bedrock needs a region, and botocore reads only AWS_DEFAULT_REGION, never
# AWS_REGION. This sets both from one resolved value so that clients built
# without an explicit region_name land in the workshop's region instead of
# whatever the active AWS profile happens to configure.
from workshop.aws_region import configure_aws_region

configure_aws_region()

import boto3

identity = boto3.client("sts").get_caller_identity()
print(f"AWS Account: {identity['Account'][-4:].rjust(12, '*')}")
print(f"Region: {boto3.session.Session().region_name}")
print("✅ AWS credentials configured")

## What is already in the graph

The restored dump holds the preloaded corpus documents, extracted under the same schema this
module pins. It ships without the vector and full-text indexes that retrieval needs. The five
held-out documents you are about to extract are also absent.

Record the current document and hotel counts, so the counts at the end of this notebook have
something to compare against. Expect fewer hotels than documents. Extraction is stochastic, so
some documents produce no complete `Hotel`, and entity resolution merges two documents that
describe the same hotel into one node.

In [ ]:
from graph_builder import connect, count_documents
from workshop.graph_connection import graph_database, require_neo4j_env

require_neo4j_env()

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        hotels_before = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]
    documents_before = count_documents(driver)

print(f"Database: {graph_database()}")
print(f"Documents already loaded: {documents_before}")
print(f"Hotels already loaded:    {hotels_before}")

### Walk both layers on a hotel from the dump

The next cell reads one hotel that arrived in the dump and reports both layers: how many chunks
its document produced, how wide the embedding on those chunks is, and how many typed facts hang
off the hotel node. The last cell in this notebook runs the same walk on the first hotel you
extract, so you can compare it against this baseline. The cell only reads.

In [ ]:
# Read-only. Reports the lexical and domain layers for one document.
LAYER_QUERY = """
MATCH (d:Document {source_filename: $filename})<-[:FROM_DOCUMENT]-(c:Chunk)
MATCH (c)<-[:FROM_CHUNK]-(h:Hotel)
WITH h, count(DISTINCT c) AS chunks, max(size(c.embedding)) AS embedding_width
RETURN h.name AS name, h.address AS address, h.guest_rating AS rating,
       chunks, embedding_width,
       count { (h)-[:HAS_ROOM]->() } AS rooms,
       count { (h)-[:OFFERS_AMENITY]->() } AS amenities,
       count { (h)-[:HAS_POLICY]->() } AS policies,
       count { (h)-[:PROVIDES_SERVICE]->() } AS services
ORDER BY name
"""


def show_both_layers(filename: str) -> None:
    """Print the lexical and domain layers for the hotels in one document."""
    with connect() as driver:
        with driver.session(database=graph_database()) as session:
            rows = session.run(LAYER_QUERY, filename=filename).data()

    if not rows:
        print(f"{filename} has no hotel in this graph.")
        return

    for row in rows:
        print(f"{row['name']}  ({filename})")
        print(f"  lexical:  {row['chunks']} chunk(s), embedding width "
              f"{row['embedding_width']}")
        print(f"  domain:   {row['rooms']} rooms, {row['amenities']} amenities, "
              f"{row['policies']} policies, {row['services']} services")
        print(f"  on the node: rating {row['rating']}, address {row['address']}")


show_both_layers("hotel-paris-001.txt")

## Load the five reserved documents

The five held-out files are the `-002` document for Tokyo, Sydney, Rio de Janeiro, Cape Town,
and Prague. They were chosen so your extraction collides with nothing:

- No later module asks a question about these five `-002` hotels, so your build cannot
  overwrite a fixture that a later module depends on.
- Each of those cities keeps its `-001` hotel in the dump, so no city drops out of the graph
  while you extract.
- Cairo is deliberately absent from the list. Module 3's hero question asks about a Cairo hotel
  by name, and that question must not depend on your extraction having succeeded.

The next cell unpacks the five files from the corpus archive and previews the first one. Read
the preview. The hotel's name, address, rating, and contact details appear as plain prose under
headings, and the rooms, amenities, policies, and services follow further down the file.
Turning that prose into typed nodes is the whole job of the extraction.

In [ ]:
from held_out_documents import HELD_OUT_DOCUMENTS, extract_held_out

paths = extract_held_out()
for path in paths:
    print(f"  {path.name}  ({path.stat().st_size:,} bytes)")

print(f"\n--- {paths[0].name}, first 400 characters ---")
print(paths[0].read_text(encoding="utf-8")[:400])

## How the extraction pipeline works

`SimpleKGPipeline` runs five stages on each document, and the workshop pins a setting at every
stage.

| Stage | Component | What it does here |
|-------|-----------|-------------------|
| Split | `FixedSizeSplitter` | Cuts the document into chunks of at most `CHUNK_SIZE` characters |
| Embed | Amazon Nova, from Setup's model table | Turns each chunk into a 1024-dimension vector and stores it on the `Chunk` node |
| Extract | Claude on Amazon Bedrock | Reads the chunk and returns JSON holding the nodes and relationships it found, restricted to the pinned schema |
| Resolve | `perform_entity_resolution=True` | Merges an extracted entity into an existing node when one already matches |
| Write | The pipeline's Neo4j writer | Creates the `Document`, `Chunk`, and entity nodes, then connects them |

**Chunk size decides what the model sees at once.** Corpus documents top out at 7,442 bytes and
`CHUNK_SIZE` is 12000, so each document becomes exactly one chunk. The hotel's name, address,
and rating therefore reach the model in the same call as its rooms and amenities. The model
never has to guess which hotel a stray amenity belongs to. `CHUNK_OVERLAP` is 0 because there
is no second chunk to overlap with. The build checks this at the end by comparing the total
chunk count against the total document count. A larger chunk count means a document was split
and its facts were extracted in two separate calls.

**One hotel per chunk produces a large answer.** A whole hotel's JSON runs past the
4096-token default that the workshop's Bedrock client sets. Past that limit the answer
truncates mid-object, and truncated JSON is
invalid, so the pipeline drops the chunk. `EXTRACTION_MAX_TOKENS` is 16000 to leave room for
the full answer.

These three settings live in `graph_config.py`, beside the build code.

**Entity resolution merges entities and never merges chunks.** That asymmetry is what lets the
build identify its own work. It records the chunk IDs in the graph before it starts, and
anything new afterwards belongs to this run.

## Why the schema is pinned

`SimpleKGPipeline` can extract without a schema. The model then chooses its own labels for each
chunk, and it chooses them from that document's own headings, which read differently in every
file. Runs without a schema produced all of these:

| Kind of drift | Labels the model chose | Why it breaks queries |
|---------------|------------------------|-----------------------|
| A property promoted to a node | `Address`, `Fee`, `Location` | The address sits on the hotel node in one document and one hop away in the next |
| A type split from its instance | `RoomType`, `BedConfiguration` | A room's own properties become separate nodes to join through |
| Two names for one thing | `ContactMethod`, `ContactInfo` | Both are reasonable, and a query has to know which one a given document used |
| Geography expanded into a hierarchy | `City`, `Country` | The city is text inside the address in most documents and a node in a few |

Each of these models the source document faithfully. Together they are unqueryable, because no
single Cypher pattern matches all of them. A better prompt cannot fix this. The vocabulary
has to be settled once, before extraction, and applied to every document.

The pinned schema does that. It names five node types, four relationship types, and the four
patterns those types may form. It also sets `additional_node_types`,
`additional_relationship_types`, and `additional_patterns` to `False`, so the model has to drop
a fact rather than invent a label for it.

The property descriptions in the schema are instructions the model reads:

- `address` says "Never model the address as its own node", which keeps `Address` out of the
  graph.
- `guest_rating` says to read `4.6` from `4.6/5.0`, which turns the document's text into a
  float. Later modules average that property, and averaging works on numbers only.
- `Amenity` says to create one "only when the document says the hotel actually has it". These
  documents also state what a hotel lacks, in sentences such as "Pool facilities are not
  available at this property." Without that instruction the graph gains a pool the hotel does
  not have.

Module 3's retrieval tool tells its agent that a hotel carries `name`, `address`, and
`guest_rating` on the node itself. The pinned schema is what makes that promise keepable.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA, OFF_SCHEMA_LABELS

print("Node types the extraction is allowed to produce:")
for node_type in GRAPH_SCHEMA["node_types"]:
    properties = ", ".join(p["name"] for p in node_type.get("properties", []))
    print(f"  :{node_type['label']:<9} {properties}")

print("\nRelationships it is allowed to produce:")
for start, rel, end in GRAPH_SCHEMA["patterns"]:
    print(f"  (:{start})-[:{rel}]->(:{end})")

print("\nLabels an unpinned run invents instead, which the build treats as a failure:")
print(f"  {', '.join(OFF_SCHEMA_LABELS)}")

### Optional: Compare an extraction without a schema

Set `RUN_UNPINNED_DEMO` to `True` to extract **one** document without a schema and print the
labels the LLM creates. Match the printed labels against the four kinds of drift above. This
comparison uses model tokens, and the main build runs whether you run it or not.

The demo leaves an extra `Document`, an extra `Chunk`, and its invented label nodes in the
graph. Nothing later in the workshop reads them, and no check in the build fails because of
them.

In [ ]:
# Optional. Extracts one document with no schema and reports what it invented.
RUN_UNPINNED_DEMO = False

if RUN_UNPINNED_DEMO:
    from graph_builder import clear_document, session as build_session, snapshot_chunk_ids
    from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import (
        FixedSizeSplitter,
    )
    from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

    from graph_config import CHUNK_OVERLAP, CHUNK_SIZE
    from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM

    sample = paths[0]
    driver = connect()
    try:
        baseline = snapshot_chunk_ids(driver)
        unpinned = SimpleKGPipeline(
            llm=BedrockLLM(),
            driver=driver,
            embedder=BedrockEmbeddings(),
            schema=None,  # the whole point
            text_splitter=FixedSizeSplitter(
                chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
            ),
            from_pdf=False,
            perform_entity_resolution=False,
        )
        await unpinned.run_async(
            file_path=sample.name,
            text=sample.read_text(encoding="utf-8"),
        )
        new_chunks = list(snapshot_chunk_ids(driver) - baseline)
        with build_session(driver) as neo4j_session:
            invented = neo4j_session.run(
                """
                MATCH (c:Chunk)<-[:FROM_CHUNK]-(n)
                WHERE elementId(c) IN $ids
                UNWIND [l IN labels(n) WHERE NOT l STARTS WITH '__'] AS label
                RETURN label, count(*) AS count ORDER BY count DESC
                """,
                ids=new_chunks,
            ).values()
        print(f"Unpinned extraction of {sample.name} produced: {invented}")
    finally:
        # Removed before the real build, so the drift demo cannot pollute it.
        clear_document(driver, sample.name)
        driver.close()
else:
    print("Skipped. Set RUN_UNPINNED_DEMO = True to watch the labels drift.")

## Build

`run_additive_build` adds the five documents to the existing graph. It runs these steps in
order:

1. Clears any earlier copy of these five documents, and nothing else.
2. Records the document count and the current chunk IDs, so the run can identify its own work.
3. Extracts the five documents one at a time, with a 180-second limit on each.
4. Retries the failures once. It clears a document again before retrying, unless a write for
   that document started in the last 30 seconds.
5. Checks that the schema held, creates the two retrieval indexes, and checks the graph
   requirements the later modules depend on.

Step 1 is what makes this cell safe to re-run. The pipeline writes with `CREATE` rather than
`MERGE`, so a plain retry would leave a second `Document` and a second `Chunk` behind for the
same file. The clear is keyed on `source_filename` and scoped to your five names, so the
preloaded documents stay untouched.

Bedrock throttles when a room full of participants extracts at the same time. The AWS client
retries with adaptive backoff, which slows every client down instead of having them all retry
into the same regional quota at full speed. If a document still fails after that, re-run this
cell.

The final step checks three things, and each one fails loudly rather than leaving a graph that
looks fine:

- **The schema held.** It lists every label this run's own chunks produced and fails if an
  off-schema label appears.
- **The indexes match the retrieval contract.** It reads both indexes back and compares type,
  state, label, property, dimensions, and similarity function.
- **The graph answers the later modules' questions.** It runs those queries now, such as Paris
  hotels with ratings and Cairo hotels that have a spa, a pool, and a rating.

The counts are strict. All five documents have to land, and each has to produce exactly one
chunk. The fixture gates are lenient about individual properties on purpose. They ask for at
least one Cairo hotel with a spa, a pool, and a rating, and at least two Paris hotels with a
rating, rather than demanding a perfect extraction from every document. Extraction is
stochastic, so one document missing one property is a bad roll. An off-schema label is
different. It means the vocabulary failed, and the build stops.

Expect about four minutes. Each document prints a line when it lands.

In [ ]:
from graph_builder import run_additive_build

exit_code = await run_additive_build(paths, "Module 1: building your five hotels")

if exit_code != 0:
    raise RuntimeError(
        "The build did not finish cleanly. Read the output above, then re-run "
        "this cell; it clears only your five documents before retrying."
    )

## The two retrieval indexes

The build creates both indexes at the end, over every `Chunk` in the graph. That includes the
chunks your extraction just wrote.

| Index | What it reads | What it finds |
|-------|---------------|---------------|
| `hotel_chunk_embeddings` | `Chunk.embedding`, cosine similarity over 1024 dimensions | Text that means the same thing as the question in different words |
| `hotel_chunk_fulltext` | `Chunk.text`, full-text | Exact strings that embeddings blur together, such as a postal code or a hotel name |

Both indexes earn their place. An embedding of `60611` sits near every other five-digit number,
so vector search can rank the right chunk low. Keyword search matches `60611` exactly. A
question phrased in words the document never uses has no keyword to match, and vector search
finds it anyway. Module 3 combines the two and lets you weight between them.

One rule holds the vector index together. The model that wrote the embeddings and the model
that embeds a query at read time have to be the same model, at the same width, for the same
purpose. A read path at a different width returns wrong rows instead of an error, which is why
the workshop's embedder takes no environment override. Its model and width are set in code.

Index creation uses `IF NOT EXISTS` and then waits for both indexes to come online, so the step
is safe to repeat and no later retrieval races the index build. The next cell reads the two
indexes back out of the database.

In [ ]:
# Read-only. Reports the two indexes the build just created.
from workshop.retrieval_contract import CHUNK_FULLTEXT_INDEX, CHUNK_VECTOR_INDEX

INDEX_QUERY = """
SHOW INDEXES YIELD name, type, state, labelsOrTypes, properties, options
WHERE name IN $names
RETURN name, type, state, labelsOrTypes, properties, options
ORDER BY name
"""

with connect() as driver:
    with driver.session(database=graph_database()) as session:
        indexes = session.run(
            INDEX_QUERY,
            names=[CHUNK_VECTOR_INDEX, CHUNK_FULLTEXT_INDEX],
        ).data()

if not indexes:
    print("Neither retrieval index exists yet. Run the build cell above.")

for index in indexes:
    config = (index["options"] or {}).get("indexConfig", {})
    labels = ", ".join(index["labelsOrTypes"])
    properties = ", ".join(index["properties"])
    print(index["name"])
    print(f"  type:       {index['type']}")
    print(f"  state:      {index['state']}")
    print(f"  indexes:    :{labels}({properties})")
    if index["type"] == "VECTOR":
        print(f"  dimensions: {config.get('vector.dimensions')}")
        print(f"  similarity: {config.get('vector.similarity_function')}")

## Query what you built

The next cell lists the five hotels you extracted with their addresses, ratings, and amenity
counts, then compares the document and hotel counts from before and after the build.

Read the rating column. Those values are floats on the `Hotel` nodes, which is what lets a
later query average them instead of estimating an average from text. Read the amenity counts
too. An amenity node exists only where a document affirmed the amenity, so counting them is
arithmetic rather than a guess.

The cell after that walks both layers for the first hotel you extracted, the same way you did
earlier for a hotel from the dump.

In [ ]:
with connect() as driver:
    with driver.session(database=graph_database()) as session:
        print("The hotels you just extracted:\n")
        for record in session.run(
            """
            MATCH (d:Document)<-[:FROM_DOCUMENT]-(:Chunk)<-[:FROM_CHUNK]-(h:Hotel)
            WHERE d.source_filename IN $filenames
            OPTIONAL MATCH (h)-[:OFFERS_AMENITY]->(a:Amenity)
            WHERE a.name IS NOT NULL
            RETURN h.name AS name, h.address AS address,
                   h.guest_rating AS rating, count(DISTINCT a) AS amenities
            ORDER BY name
            """,
            filenames=list(HELD_OUT_DOCUMENTS),
        ):
            print(f"  {record['name']}")
            print(f"    {record['address']}")
            print(f"    rating {record['rating']}, {record['amenities']} amenities\n")

        hotels_after = session.run("MATCH (h:Hotel) RETURN count(h) AS n").single()["n"]

    documents_after = count_documents(driver)

print(f"Documents: {documents_before} -> {documents_after}")
print(f"Hotels:    {hotels_before} -> {hotels_after}")
print(
    "\nEvery aggregation and multi-hop question from here on runs across the whole "
    "graph, yours included."
)

In [ ]:
# Read-only. The same both-layers walk, now on a hotel you extracted yourself.
show_both_layers(HELD_OUT_DOCUMENTS[0])

## What is a Strands agent?

Module 2 creates two agents with the Strands Agents SDK. Three pieces do that work: the
`Agent` class, the model ID you pin on it, and the `@tool` decorator.

**`Agent` runs the agent loop.** You configure it with a model, a system prompt, and a list of
tools. The `Agent` sends the user's message to the model. When the model requests a tool, the
`Agent` runs it and returns the result to the model. The loop ends when the model gives a final
answer.

```python
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-sonnet-5"),
    system_prompt="Answer only from tool results. If a tool returns nothing, say so.",
    tools=[search_hotels],
)
```

**Use a fixed model.** `BedrockModel` accepts the same model ID that you would pass to `boto3`.
A fixed model ID ensures that you and the facilitator use the same model when comparing outputs
in Module 2.

**`@tool` exposes a Python function to the model.** The model reads the function's docstring to
decide when to use the tool.

```python
from strands import tool

@tool
def search_hotels(question: str) -> str:
    """Search hotel knowledge. Use for questions about specific hotels."""
    return retriever.search(question)
```

**Key point:** Good grounding requires accurate tool results. If a tool returns plausible text
that fails to answer the question, the agent can give a confident but incorrect answer. A system
prompt alone does not reliably prevent this problem. Module 2 demonstrates this failure, and
Module 3 fixes it.

In [ ]:
from workshop.workshop_utils import lego_progress

lego_progress(1)